# OR-R1 Small Batch Experiment

这个 notebook 用来自己跑一小批 OR-R1 实验，重点看三件事：

1. `SFT` 样本到底长什么样
2. `GRPO` 一次采样出来的多个 completion 是什么
3. reward 是怎么从格式、代码执行、vote 组合起来的

建议先把 kernel 切到仓库虚拟环境：`/mnt/workspace0/WWWWWWWWWWWW/OR-R1/.venv/bin/python`。

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json
import sys

import pandas as pd
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

REPO_ROOT = Path('/mnt/workspace0/WWWWWWWWWWWW/OR-R1')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from or_r1_flow_trace import load_json_or_jsonl, build_sft_trace, trace_grpo, run_code, format_reward_of, safe_int_vote, relative_match
from structure_reward import build_completion_schema, build_problem_schema, score_structure
from utils.data import encode_with_prompt_completion_format
from eval.self_repair_pipeline_v2 import run_code_v2, diagnosis_from_row_v2, build_feedback_v2, acceptance_tier, prompt_for_route

MODEL_PATH = REPO_ROOT / 'output' / 'sft_qwen3_8b_dir_3Ksample_1epoch'
SFT_DATASET_PATH = REPO_ROOT / 'datasets' / 'OR-Instruct-Data-3K' / 'OR-Instruct-Data-1.json'
GRPO_DATASET_PATH = REPO_ROOT / 'datasets' / 'trainset' / 'train_100.jsonl'
WORK_DIR = REPO_ROOT / 'output' / 'notebook_small_batch'
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('repo:', REPO_ROOT)
print('model:', MODEL_PATH)
print('work_dir:', WORK_DIR)

## 0.5 Notebook Runtime Helpers

`trace_grpo(...)` 默认走 vLLM，多卡初始化更快，但在 notebook 里也更容易因为显存残留或 kernel 解释器不一致而报 `WorkerProc initialization failed`。

下面这个 helper 会：

- 先尝试 vLLM
- 如果失败，自动回退到 `transformers.generate` 的小批量路径

这样你至少能继续做小实验，不会被 vLLM 初始化卡死。

In [ ]:
def release_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def _reward_rows_from_completions(completions, gt_answer):
    rows = []
    prediction_answers = []
    for idx, completion in enumerate(completions):
        fr, markers_hit = format_reward_of(completion)
        exec_output = run_code(completion)
        valid_code_reward = 0.0
        prediction = None
        if exec_output is not None:
            prediction = exec_output['execution_best_solution']
            if prediction is not None:
                valid_code_reward = 1.0
        prediction_answers.append(prediction)
        rows.append({
            'completion_index': idx,
            'completion_chars': len(completion),
            'completion_preview': completion[:2200],
            'format_reward': fr,
            'format_markers_hit': markers_hit,
            'valid_code_reward': valid_code_reward,
            'prediction_answer': prediction,
            'execution_state': None if exec_output is None else exec_output['execution_state'],
            'execution_result_tail': None if exec_output is None else exec_output['execution_result'][-1200:],
            'script_preview': None if exec_output is None else exec_output['script'][:2200],
        })

    vote_counts = {}
    for prediction in prediction_answers:
        if prediction is None or prediction == 'No Best Solution':
            continue
        voted = safe_int_vote(prediction)
        if voted is None:
            continue
        vote_counts[voted] = vote_counts.get(voted, 0) + 1

    voting_answer = None
    max_count = 1
    for key, count in vote_counts.items():
        if count > max_count:
            max_count = count
            voting_answer = key

    for row in rows:
        prediction = row['prediction_answer']
        if prediction is None or prediction == 'No Best Solution' or voting_answer is None:
            answer_reward = 0.0
        else:
            voted = safe_int_vote(prediction)
            answer_reward = 1.0 if voted == voting_answer else 0.0
        row['answer_reward'] = answer_reward
        row['total_reward'] = row['format_reward'] + row['valid_code_reward'] + answer_reward
        row['gt_answer'] = gt_answer
        row['matches_gt'] = relative_match(prediction, gt_answer)

    return rows, voting_answer, vote_counts


def trace_grpo_fallback_transformers(model_path, tokenizer, sample, args):
    prompt_body = (
        'Below is an operations research question. Build a mathematical model and corresponding python code using `coptpy` that appropriately addresses the question.\n\n'
        f"# Question:\n{sample['question'].strip()}\n\n# Response:"
    )
    prompt = tokenizer.apply_chat_template([{'role': 'user', 'content': prompt_body}], tokenize=False)
    prompt_inputs = tokenizer(prompt, return_tensors='pt').to('cuda' if torch.cuda.is_available() else 'cpu')

    gen_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    ).to(prompt_inputs['input_ids'].device)
    gen_model.eval()

    completions = []
    for _ in range(args.num_generations):
        with torch.no_grad():
            outputs = gen_model.generate(
                **prompt_inputs,
                do_sample=True,
                temperature=args.temperature,
                top_p=args.top_p,
                max_new_tokens=args.max_tokens,
                pad_token_id=tokenizer.eos_token_id,
            )
        full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
        completions.append(full_text[len(prompt):])

    rows, voting_answer, vote_counts = _reward_rows_from_completions(completions, sample['answer'])
    trace = {
        'question_preview': sample['question'][:2400],
        'gt_answer': sample['answer'],
        'prompt_body_preview': prompt_body[:2200],
        'chat_prompt_preview': prompt[:2200],
        'prompt_tokens': int(prompt_inputs['input_ids'].shape[1]),
        'num_generations': args.num_generations,
        'temperature': args.temperature,
        'top_p': args.top_p,
        'voting_answer': voting_answer,
        'voting_counts': vote_counts,
        'rows': rows,
        'backend': 'transformers_fallback',
    }

    del gen_model
    del prompt_inputs
    del outputs
    release_cuda()
    return trace


def safe_trace_grpo(model_path, tokenizer, sample, args):
    try:
        trace = trace_grpo(model_path, tokenizer, sample, args)
        trace['backend'] = 'vllm'
        return trace
    except Exception as exc:
        print('vLLM trace_grpo failed, fallback to transformers.generate')
        print('root error:', repr(exc))
        return trace_grpo_fallback_transformers(model_path, tokenizer, sample, args)

## 1. 看一个真实 SFT 样本

这里看的是 `prompt/completion` 监督样本。SFT 训练时，模型就是在学：给定 `prompt`，怎样把 `completion` 写出来。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
sft_dataset = load_json_or_jsonl(str(SFT_DATASET_PATH))
sft_sample = sft_dataset[0]
sft_trace = build_sft_trace(tokenizer, sft_sample)

pd.DataFrame([
    {
        'prompt_chars': sft_trace['prompt_chars'],
        'completion_chars': sft_trace['completion_chars'],
        'prompt_tokens': sft_trace['prompt_tokens'],
        'full_tokens': sft_trace['full_tokens'],
        'completion_tokens_effective': sft_trace['completion_tokens_effective'],
    }
])

In [ ]:
print('=== SFT prompt preview ===')
print(sft_trace['prompt_preview'])

print('\n=== SFT completion preview ===')
print(sft_trace['completion_preview'])

## 1.5 SFT 真正的训练过程

上面只是看原始 `prompt/completion`。SFT 真正训练时，会先把它们拼起来，然后把 `prompt` 对应的 label 全部 mask 成 `-100`，只在 `completion` token 上算 loss。

In [ ]:
sft_encoded = encode_with_prompt_completion_format(
    sft_sample,
    tokenizer=tokenizer,
    max_seq_length=4096,
)

input_ids = sft_encoded['input_ids']
labels = sft_encoded['labels']
supervised_mask = labels != -100

pd.DataFrame([
    {
        'total_tokens': int(input_ids.shape[0]),
        'masked_prompt_tokens': int((labels == -100).sum().item()),
        'supervised_completion_tokens': int(supervised_mask.sum().item()),
        'first_supervised_token_index': int(torch.where(supervised_mask)[0][0].item()),
    }
])

In [ ]:
supervised_ids = input_ids[supervised_mask]
masked_ids = input_ids[~supervised_mask]

print('=== masked prompt part ===')
print(tokenizer.decode(masked_ids[:400], skip_special_tokens=False))

print('\n=== supervised completion part ===')
print(tokenizer.decode(supervised_ids[:600], skip_special_tokens=False))

In [ ]:
# 这一步是一个真实的 SFT forward：
# 模型吃的是整条 input_ids，但 loss 只会在 labels != -100 的位置计算。
# 第一次加载 8B 模型会比较慢，也比较占显存。

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if device == 'cuda' else torch.float32,
).to(device)
model.eval()

batch = {
    'input_ids': input_ids.unsqueeze(0).to(device),
    'attention_mask': sft_encoded['attention_mask'].unsqueeze(0).to(device),
    'labels': labels.unsqueeze(0).to(device),
}

with torch.no_grad():
    outputs = model(**batch)
    sft_forward_loss = float(outputs.loss)

print('sft_forward_loss =', sft_forward_loss)

# 关键：这一格跑完后立刻释放 8B 模型，否则后面的 GRPO trace 会再加载一份模型，24G 显存会直接爆掉。
del outputs
del batch
del model
release_cuda()

### SFT forward 后的显存清理

如果你重复运行前面的 cell，或者中途改过模型加载代码，可以手动再跑一次这一格，确保 GPU 没残留模型。

In [ ]:
for name in ['model', 'outputs', 'batch', 'gen_model', 'prompt_inputs']:
    if name in globals():
        del globals()[name]

release_cuda()
print('CUDA cache released')

## 2. 准备一个小 GRPO 实验样本

先用一个短题，方便快速看到 8 个 completion 和 reward 表。你也可以把 `easy_question` 换成自己的题。

In [ ]:
easy_question = {
    'question': (
        'An ecotourism company plans to carry out four projects (x1, x2, x3, x4). '
        'Their environmental impact indices are 20, 30, 40, and 50, and their revenues are '
        '10000, 20000, 30000, and 40000 respectively. The total environmental impact cannot exceed '
        '100, and at most three projects can be selected. Build an optimization model to maximize '
        'total revenue and return the optimal objective value.'
    ),
    'answer': 70000.0,
}

easy_jsonl = WORK_DIR / 'grpo_easy_sample.jsonl'
with easy_jsonl.open('w', encoding='utf-8') as f:
    f.write(json.dumps(easy_question, ensure_ascii=False) + '\n')

easy_jsonl

## 3. 跑一次小批量 GRPO trace

这一步会真正加载模型，并且：

- 对同一个问题采样 `num_generations` 个 completion
- 尝试抽取代码并执行
- 计算 `format_reward / valid_code_reward / answer_reward`

第一次跑会慢，主要时间花在模型加载。

In [ ]:
args = SimpleNamespace(
    tensor_parallel_size=1,
    gpu_memory_utilization=0.35,
    max_model_len=8192,
    num_generations=2,
    temperature=0.8,
    top_p=0.95,
    max_tokens=512,
)

grpo_dataset = load_json_or_jsonl(str(easy_jsonl))
grpo_sample = grpo_dataset[0]
grpo_trace = safe_trace_grpo(str(MODEL_PATH), tokenizer, grpo_sample, args)
print('backend =', grpo_trace.get('backend'))

In [ ]:
reward_df = pd.DataFrame([
    {
        'idx': row['completion_index'],
        'prediction_answer': row['prediction_answer'],
        'execution_state': row['execution_state'] if row['execution_state'] is not None else 'no_code',
        'format_reward': row['format_reward'],
        'valid_code_reward': row['valid_code_reward'],
        'answer_reward': row['answer_reward'],
        'total_reward': row['total_reward'],
        'matches_gt': row['matches_gt'],
    }
    for row in grpo_trace['rows']
])

print('gt_answer =', grpo_trace['gt_answer'])
print('voting_answer =', grpo_trace['voting_answer'])
print('voting_counts =', grpo_trace['voting_counts'])
reward_df

In [ ]:
row_id = 0
row = grpo_trace['rows'][row_id]

print('=== completion preview ===')
print(row['completion_preview'])

print('\n=== execution state ===')
print(row['execution_state'])

print('\n=== execution result tail ===')
print(row['execution_result_tail'])

## 4. 换成一个真实 train_100 长题

这个 cell 对应我们之前 trace 里看到的情况：prompt 很长，reward 经常退化成格式分。

默认取 `train_100` 里的第 98 条。

In [ ]:
trainset = load_json_or_jsonl(str(GRPO_DATASET_PATH))
long_sample = trainset[98]
print('question chars =', len(long_sample['question']))
print('gt answer =', long_sample['answer'])
print(long_sample['question'][:2000])

In [ ]:
args_long = SimpleNamespace(
    tensor_parallel_size=1,
    gpu_memory_utilization=0.35,
    max_model_len=8192,
    num_generations=2,
    temperature=0.8,
    top_p=0.95,
    max_tokens=384,
)

long_trace = safe_trace_grpo(str(MODEL_PATH), tokenizer, long_sample, args_long)
print('backend =', long_trace.get('backend'))
pd.DataFrame([
    {
        'idx': row['completion_index'],
        'prediction_answer': row['prediction_answer'],
        'execution_state': row['execution_state'] if row['execution_state'] is not None else 'no_code',
        'format_reward': row['format_reward'],
        'valid_code_reward': row['valid_code_reward'],
        'answer_reward': row['answer_reward'],
        'total_reward': row['total_reward'],
    }
    for row in long_trace['rows']
])

## 5. Structure Reward 小批量实验

这一页专门看 `r_struct`。你会看到：

- 题目文本会先被抽成 `problem_schema`
- completion 会被抽成 `completion_schema`
- 然后拆成 `r_obj / r_var / r_con / r_align / r_struct`

先用两个自造 completion 看分数差异。

In [ ]:
struct_problem = easy_question['question']

struct_good_completion = '''
## Mathematical Model:
Use binary variables x_i to decide whether each project is selected.

## Decision Variables:
x_i is binary for i in {1,2,3,4}.

## Objective Function:
Maximize 10000 x_1 + 20000 x_2 + 30000 x_3 + 40000 x_4.

## Constraints:
20 x_1 + 30 x_2 + 40 x_3 + 50 x_4 <= 100.
x_1 + x_2 + x_3 + x_4 <= 3.
x_i in {0,1}.

## Python Code Solution Using `coptpy`:
```python
import coptpy as cp
from coptpy import COPT
env = cp.Envr()
model = env.createModel('p')
```
'''

struct_bad_completion = '''
## Mathematical Model:
Use continuous variables y_i to represent fractional resource levels.

## Decision Variables:
y_i are continuous.

## Objective Function:
Minimize y_1 + y_2 + y_3 + y_4.

## Constraints:
y_i >= 0.

## Python Code Solution Using `coptpy`:
```python
print('placeholder')
```
'''

struct_cases = [
    ('good_like', struct_good_completion),
    ('bad_like', struct_bad_completion),
]

pd.DataFrame([
    {
        'case': name,
        **score_structure(struct_problem, completion),
    }
    for name, completion in struct_cases
])

In [ ]:
problem_schema = build_problem_schema(struct_problem)
completion_schema = build_completion_schema(struct_problem, struct_good_completion)

print('=== problem schema ===')
print(problem_schema)

print('\n=== completion schema (good_like) ===')
print(completion_schema)

In [ ]:
# 可选：如果你已经跑过 grpo_trace / long_trace，也可以直接给真实 completion 打 structure reward。
trace_obj = long_trace if 'long_trace' in globals() else grpo_trace
pd.DataFrame([
    {
        'idx': row['completion_index'],
        **score_structure(trace_obj['question_preview'], row['completion_preview']),
    }
    for row in trace_obj['rows']
])

## 6. Self-Repair 小批量实验

这一页专门看一个失败样本怎样经过：

- `initial_response`
- `run_code_v2`
- `diagnosis_from_row_v2`
- `build_feedback_v2`
- `prompt_for_route`
- `acceptance_tier`

先用一个故意失败的例子，保证 notebook 自包含。

In [ ]:
repair_row = {
    'question': easy_question['question'],
    'answer': easy_question['answer'],
    'initial_response': '''
## Mathematical Model:
Choose binary project variables.

## Decision Variables:
x_i binary.

## Objective Function:
Maximize revenue.

## Constraints:
Impact <= 100, at most 3 projects.

## Python Code Solution Using `coptpy`:
```python
import coptpy as cp
from coptpy import COPT
env = cp.Envr()
model = env.createModel('broken')
x = model.addVar(vtype=COPT.BINARY, name='x')
model.setObjective(revenue * x, COPT.MAXIMIZE)
model.solve()
```
'''.strip(),
}

initial_exec = run_code_v2(repair_row['initial_response'], timeout=30)
repair_row.update({
    'initial_execution_state': initial_exec.state,
    'initial_execution_best_solution': initial_exec.best_solution,
    'initial_execution_stdout': initial_exec.stdout,
    'initial_execution_stderr': initial_exec.stderr,
})
repair_row['diagnosis'] = diagnosis_from_row_v2(repair_row, tolerance=0.05)
repair_row['coarse_feedback'] = build_feedback_v2(repair_row, initial_exec)

pd.DataFrame([{
    'initial_execution_state': repair_row['initial_execution_state'],
    'initial_execution_best_solution': repair_row['initial_execution_best_solution'],
    **repair_row['diagnosis'],
}])

In [ ]:
print('=== initial response ===')
print(repair_row['initial_response'])

print('\n=== execution stdout ===')
print(repair_row['initial_execution_stdout'])

print('\n=== execution stderr ===')
print(repair_row['initial_execution_stderr'])

print('\n=== diagnosis ===')
print(json.dumps(repair_row['diagnosis'], ensure_ascii=False, indent=2))

print('\n=== feedback ===')
print(repair_row['coarse_feedback'])

In [ ]:
diagnosis_prompt = prompt_for_route(tokenizer, repair_row, kind='diagnosis')
coarse_prompt = prompt_for_route(tokenizer, repair_row, kind='coarse')

print('=== diagnosis prompt preview ===')
print(diagnosis_prompt[:4000])

print('\n=== coarse prompt preview ===')
print(coarse_prompt[:3000])

In [ ]:
# acceptance_tier 需要 repair 后的结果。这里手动伪造一个“修好”的 after-state，演示它是怎样分 tier 的。
repair_row_demo = dict(repair_row)
repair_row_demo['diagnosis_execution_state'] = 'optimal'
repair_row_demo['diagnosis_execution_best_solution'] = '70000'
tier = acceptance_tier(repair_row_demo, tolerance=0.05)
print('acceptance_tier =', tier)

## 7. 你可以自己改的几个关键开关

- `num_generations`: 一次采样几个候选。原始 OR-R1 是 8。
- `max_tokens`: completion 最长生成多少 token。
- `temperature`: 采样随机性。
- `grpo_sample`: 换题。
- `struct_good_completion / struct_bad_completion`: 换不同 completion 看结构分。
- `repair_row['initial_response']`: 换不同失败输出，看 self-repair diagnosis 和 route 如何变化。

如果你只想看数据格式，不想加载模型，跑到第 2 部分就够了。
如果你要看 reward 真正怎么打，跑第 3/4 部分。
如果你要看两个增强方向，跑第 5/6 部分。